In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import f_classif
import xgboost as xgb

file_path = 'path/train.xlsx'
df = pd.read_excel(file_path, sheet_name='Sheet1')
df.columns = df.columns.astype(str)

glioma_df = df[df['Label'] == 1]
normal_df = df[df['Label'] == 0]

print("Glioma:", len(glioma_df))
print("Normal:", len(normal_df))

feature_names = df.drop('Label', axis=1).columns
n_features = len(feature_names)

n_repeat = 1000
top_k = 5

rf_all = []
xgb_all = []
anova_all = []

rf_topk_count = np.zeros(n_features)
xgb_topk_count = np.zeros(n_features)
anova_topk_count = np.zeros(n_features)

for i in range(n_repeat):

    print(f"Repeat {i+1}/{n_repeat}")
    n_sample = len(normal_df)
    glioma_sample = glioma_df.sample(n=n_sample, random_state=i)
    balanced_df = pd.concat([glioma_sample, normal_df], axis=0)
    balanced_df = balanced_df.sample(frac=1, random_state=i)
    X = balanced_df.drop('Label', axis=1)
    y = balanced_df['Label']

    rf = RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
    rf.fit(X, y)
    rf_imp = rf.feature_importances_
    rf_all.append(rf_imp)
    rf_topk_count[np.argsort(rf_imp)[-top_k:]] += 1

    xgb_model = xgb.XGBClassifier(
        random_state=42,
        eval_metric='logloss'
    )
    xgb_model.fit(X, y)
    xgb_imp = xgb_model.feature_importances_
    xgb_all.append(xgb_imp)
    xgb_topk_count[np.argsort(xgb_imp)[-top_k:]] += 1

    f_values, _ = f_classif(X, y)
    anova_all.append(f_values)
    anova_topk_count[np.argsort(f_values)[-top_k:]] += 1

rf_all = np.array(rf_all)
xgb_all = np.array(xgb_all)
anova_all = np.array(anova_all)

def mean_std(data):
    mean = data.mean(axis=0)
    std = data.std(axis=0)
    return mean, std

rf_mean, rf_std = mean_std(rf_all)
xgb_mean, xgb_std = mean_std(xgb_all)
anova_mean, anova_std = mean_std(anova_all)

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'RF_mean': rf_mean, 'RF_std': rf_std, 'RF_topk_freq': rf_topk_count / n_repeat,
    'XGB_mean': xgb_mean, 'XGB_std': xgb_std, 'XGB_topk_freq': xgb_topk_count / n_repeat,
    'ANOVA_mean': anova_mean, 'ANOVA_std': anova_std, 'ANOVA_topk_freq': anova_topk_count / n_repeat})

cols = ['RF_mean', 'XGB_mean', 'ANOVA_mean']

for col in cols:
    max_val = importance_df[col].max()
    if max_val > 0:
        importance_df[col] = importance_df[col] / max_val

importance_df['Average_Score'] = importance_df[cols].mean(axis=1)

importance_df = importance_df.sort_values(
    by='Average_Score',
    ascending=False
)
importance_df.to_csv("path\importance.csv", index=False, encoding='utf-8-sig')